## Notebook 14 - Match-level centrality distribution: entropy and $N_{eff}$ vs result
Núria Pascual Salas

**Content:** Tests whether a team's match-level distribution of player influence (how
spread the PageRank is among its players in a single game) relates to that game's
result. This is the match-level counterpart of notebook 13, which worked on
consolidated season networks. The unit of analysis is the team-match (≈760
observations), not the player: each team-match yields one entropy $S$ and one
$N_{\mathrm{eff}}$. Since a team fields only ~11-16 players per match, the normalised
measures $S$ (over $\ln N$) and $N_{\mathrm{eff}}/N$ are used to control for roster
size, as in notebook 13. Two analyses follow: a bivariate comparison across
win/draw/loss, and a logistic regression for P(win) testing whether $S$ adds anything
over team SF.

**Outputs:**
- outputs/csv/match_entropy_neff.csv
- outputs/csv/match_logit_results.csv

**Used in:** Chapter 5, Section 5.4 (match-level entropy and result).

In [1]:
from utils import *
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
from sklearn.metrics import roc_auc_score, accuracy_score
import os

FIGURES_DIR = 'outputs/figures'
CSV_DIR     = 'outputs/csv'
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

### 1. Build the team-match table

In [2]:
rows = []
print('Building team-match distribution descriptors...')
for team_id, team_name in all_teams.items():
    n_matches = 0
    for m_id, events in stream_matches_from_zip(zip_path, folder_laliga, '_events.json'):
        teams_in_match = {e['team']['id'] for e in events if 'team' in e}
        if team_id not in teams_in_match:
            continue

        G = build_passing_network(events, team_name)
        _, _, result, _ = get_match_result(events, team_id)
        if G.number_of_nodes() < 5 or result is None:
            continue

        sf_value = calculate_match_sf(events, team_id)
        pr = nx.pagerank(G, weight='weight')
        N, S, N_eff, N_eff_frac = entropy_neff(list(pr.values()))

        rows.append({
            'team':       team_name,
            'team_id':    team_id,
            'match_id':   m_id,
            'outcome':    result,                      # 'win' / 'draw' / 'loss'
            'won':        1 if result == 'win' else 0,
            'N':          N,
            'S':          S,
            'N_eff':      N_eff,
            'N_eff_frac': N_eff_frac,
            'team_sf':    sf_value,
        })
        n_matches += 1
    print(f'  {team_name}: {n_matches} matches')

df = pd.DataFrame(rows).dropna(subset=['S', 'team_sf'])
print(f'\nTotal team-match observations: {len(df)}')
print(df['outcome'].value_counts())
df.to_csv(f'{CSV_DIR}/match_entropy_neff.csv', index=False)

Building team-match distribution descriptors...
  Deportivo Alavés: 38 matches
  Granada: 38 matches
  Barcelona: 38 matches
  Almería: 38 matches
  Sevilla: 38 matches
  Cádiz: 38 matches
  Girona: 38 matches
  Athletic Club: 38 matches
  Real Sociedad: 38 matches
  Mallorca: 38 matches
  Real Betis: 38 matches
  Atlético Madrid: 38 matches
  Villarreal: 38 matches
  Celta Vigo: 38 matches
  Valencia: 38 matches
  Las Palmas: 38 matches
  Osasuna: 38 matches
  Real Madrid: 38 matches
  Getafe: 38 matches
  Rayo Vallecano: 38 matches

Total team-match observations: 760
outcome
win     277
lost    277
draw    206
Name: count, dtype: int64


### 2. Descriptive: distribution descriptors by outcome

In [3]:
summary = df.groupby('outcome')[['S', 'N_eff', 'N_eff_frac', 'N']].agg(
    ['mean', 'std', 'count']).round(3)
print(summary)

             S                N_eff              N_eff_frac               \
          mean    std count    mean    std count       mean    std count   
outcome                                                                    
draw     0.953  0.013   206  12.378  0.861   206      0.805  0.044   206   
lost     0.955  0.012   277  12.612  0.827   277      0.806  0.041   277   
win      0.950  0.013   277  12.291  0.863   277      0.794  0.045   277   

              N               
           mean    std count  
outcome                       
draw     15.374  0.759   206  
lost     15.643  0.685   277  
win      15.487  0.750   277  


### 3. Bivariate tests: does the distribution differ across W / D / L?

In [4]:
def cohens_d(a, b):
    n1, n2 = len(a), len(b)
    s1, s2 = np.std(a, ddof=1), np.std(b, ddof=1)
    pooled = np.sqrt(((n1-1)*s1**2 + (n2-1)*s2**2) / (n1+n2-2))
    return (np.mean(a) - np.mean(b)) / pooled

for measure in ['S', 'N_eff_frac']:
    wins   = df[df['outcome'] == 'win'][measure].values
    draws  = df[df['outcome'] == 'draw'][measure].values
    losses = df[df['outcome'] == 'lost'][measure].values
    h, p_kw = stats.kruskal(wins, draws, losses)
    u, p_mw = stats.mannwhitneyu(wins, losses, alternative='two-sided')
    d = cohens_d(wins, losses)
    mag = ('negligible' if abs(d) < 0.2 else 'small' if abs(d) < 0.5
           else 'medium' if abs(d) < 0.8 else 'large')
    print(f"--- {measure} ---")
    print(f"  mean  W={wins.mean():.4f}  D={draws.mean():.4f}  L={losses.mean():.4f}")
    print(f"  Kruskal-Wallis p = {p_kw:.4f}")
    print(f"  Mann-Whitney (W vs L) p = {p_mw:.4f}")
    print(f"  Cohen's d (W vs L) = {d:+.3f}  ({mag})\n")

--- S ---
  mean  W=0.9501  D=0.9531  L=0.9547
  Kruskal-Wallis p = 0.0000
  Mann-Whitney (W vs L) p = 0.0000
  Cohen's d (W vs L) = -0.371  (small)

--- N_eff_frac ---
  mean  W=0.7939  D=0.8054  L=0.8064
  Kruskal-Wallis p = 0.0007
  Mann-Whitney (W vs L) p = 0.0004
  Cohen's d (W vs L) = -0.288  (small)



In [5]:
print(df['outcome'].value_counts())
print(df['outcome'].unique())

outcome
win     277
lost    277
draw    206
Name: count, dtype: int64
['win' 'lost' 'draw']


### 4. Multivariate: does $S$ add anything over team_sf?

In [6]:
for col in ['S', 'N_eff_frac', 'team_sf']:
    df[f'{col}_z'] = (df[col] - df[col].mean()) / df[col].std()

m_sf   = smf.logit('won ~ team_sf_z', data=df).fit(disp=False)
m_s    = smf.logit('won ~ S_z', data=df).fit(disp=False)
m_both = smf.logit('won ~ team_sf_z + S_z', data=df).fit(disp=False)

compare = pd.DataFrame({
    'Model':      ['team_sf only', 'S only', 'team_sf + S'],
    'AIC':        [m_sf.aic, m_s.aic, m_both.aic],
    'Pseudo R2':  [m_sf.prsquared, m_s.prsquared, m_both.prsquared],
}).round(4)
print(compare.to_string(index=False))
print()
print('Full model (team_sf + S) coefficients:')
coefs = pd.DataFrame({
    'coef':    m_both.params,
    'OR':      np.exp(m_both.params),
    'p_value': m_both.pvalues,
}).round(4)
print(coefs)
coefs.to_csv(f'{CSV_DIR}/match_logit_results.csv')

       Model      AIC  Pseudo R2
team_sf only 983.4820     0.0176
      S only 984.0748     0.0170
 team_sf + S 974.5071     0.0286

Full model (team_sf + S) coefficients:
             coef      OR  p_value
Intercept -0.5778  0.5611   0.0000
team_sf_z -0.2693  0.7639   0.0008
S_z       -0.2604  0.7707   0.0010


In [7]:
y_true = df['won'].values
y_prob = m_both.predict(df)
auc = roc_auc_score(y_true, y_prob)
acc = accuracy_score(y_true, (y_prob >= 0.5).astype(int))
print(f"Full model (team_sf + S):  AUC = {auc:.3f}   Accuracy = {acc:.3f}")
print(f"Baseline P(win) = {df['won'].mean():.3f}")

Full model (team_sf + S):  AUC = 0.617   Accuracy = 0.658
Baseline P(win) = 0.364


### Summary

In [8]:
print('=' * 70)
print('SUMMARY - Match-level entropy / N_eff vs result')
print('=' * 70)
print(f'\nTeam-match observations: n = {len(df)}')
print(f"\nMatch entropy S:        mean {df['S'].mean():.3f}  (W={df[df.outcome=='win']['S'].mean():.3f}, "
      f"L={df[df.outcome=='lost']['S'].mean():.3f})")
print(f"Effective fraction:     mean {df['N_eff_frac'].mean():.3f}")
print('\n--- Model comparison (lower AIC = better) ---')
print(compare.to_string(index=False))
print(f'\nFull model AUC = {auc:.3f} (baseline P(win) = {df["won"].mean():.3f})')

SUMMARY - Match-level entropy / N_eff vs result

Team-match observations: n = 760

Match entropy S:        mean 0.953  (W=0.950, L=0.955)
Effective fraction:     mean 0.802

--- Model comparison (lower AIC = better) ---
       Model      AIC  Pseudo R2
team_sf only 983.4820     0.0176
      S only 984.0748     0.0170
 team_sf + S 974.5071     0.0286

Full model AUC = 0.617 (baseline P(win) = 0.364)
